In [1]:
!pip install pandas numpy scikit-learn matplotlib fastapi uvicorn python-multipart torch torchvision pillow mlflow

# Nova Stad - MLOps Pipeline

Dit rapport is gemaakt door Eduard, gianni en Wouter

github: https://github.com/gianni-dk/MLOps-Nova-Stad

## Inleiding

Dit notebook beschrijft de MLOps pipeline voor project Nova Stad. Het systeem voorspelt verkeersvolume op basis van historische sensordata (Cloud Model) en detecteert voertuigen en voetgangers via een lichtgewicht neuraal netwerk (Edge Model).

De pipeline is opgebouwd rondom vijf leerdoelen:

1. Data Ingestion en opslag
2. Schaalbaarheid
3. Modellering (Cloud en Edge)
4. Deployment
5. Monitoring

## 1. Monitoring (Leerdoel 5)

De `DataMonitor` class houdt bij wat er met de data gebeurt tijdens de pipeline. Elke stap wordt gelogd zodat afwijkingen snel zichtbaar zijn.

De methode `trigger_alert()` slaat basisstatistieken op in een lokaal bestand `monitoring_metrics.csv`. Dit simuleert een dashboard backend: in een productie-omgeving zou een tool zoals Grafana dit bestand uitlezen en de statistieken in een dashboard weergeven.

In [2]:
import os
import pandas as pd
import numpy as np
import logging
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error


logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)


class DataMonitor:
    """Verzorgt de logging en controleert op basale afwijkingen in de data."""

    @staticmethod
    def log_data_shape(step_name: str, df: pd.DataFrame):
        logger.info(f"[{step_name}] Data dimensies: {df.shape}")

    @staticmethod
    def check_missing_values(df: pd.DataFrame):
        missing = df.isnull().sum().sum()
        if missing > 0:
            logger.warning(f"Let op: {missing} ontbrekende waarden gedetecteerd in de dataset.")
        else:
            logger.info("Geen ontbrekende waarden gedetecteerd.")

    @staticmethod
    def trigger_alert(df: pd.DataFrame):
        """Sla basisstatistieken op in monitoring_metrics.csv."""
        stats = {
            "timestamp": [pd.Timestamp.now().isoformat()],
            "rows": [df.shape[0]],
            "columns": [df.shape[1]],
            "missing_values": [int(df.isnull().sum().sum())],
            "mean_traffic_volume": [
                round(df["traffic_volume"].mean(), 2)
                if "traffic_volume" in df.columns else None
            ]
        }
        stats_df = pd.DataFrame(stats)
        file_exists = os.path.isfile("monitoring_metrics.csv")
        stats_df.to_csv("monitoring_metrics.csv", mode="a", header=not file_exists, index=False)
        logger.info("Statistieken opgeslagen in monitoring_metrics.csv")

## 2. Data Ingestion en Preprocessing (Leerdoel 1 & 2)

### Data Ingestion via URL (Leerdoel 1)

De dataset wordt rechtstreeks ingeladen via een URL, niet via een lokaal bestand. Dit is een bewuste MLOps-keuze: in een geautomatiseerde pipeline kan het systeem op elk moment de meest recente data ophalen zonder handmatige tussenkomst. Het werkt ook in een CI/CD-omgeving of op een cloudserver waar lokale bestanden niet beschikbaar zijn.

### Schaalbaarheid via chunksize (Leerdoel 2)

De `load_data` functie leest de data in met `chunksize=10000`. Pandas laadt de dataset dan in stukken van 10.000 rijen en voegt deze daarna samen. Bij kleine datasets merk je hier weinig van, maar bij datasets van meerdere gigabytes voorkomt dit dat het werkgeheugen volloopt. Het systeem werkt daardoor met veel grotere datasets zonder aanpassingen aan de code.

In [3]:
class DataIngestor:
    """Verantwoordelijk voor het inladen van de dataset via URL."""

    def __init__(self, url: str):
        self.url = url

    def load_data(self) -> pd.DataFrame:
        logger.info("Start inladen van data via URL...")
        chunks = pd.read_csv(self.url, chunksize=10000)
        df = pd.concat(chunks, ignore_index=True)
        DataMonitor.log_data_shape("Data Ingestion", df)
        DataMonitor.trigger_alert(df)
        return df


class Preprocessor:
    """Verantwoordelijk voor het opschonen en transformeren van data."""

    def process(self, df: pd.DataFrame) -> tuple:
        logger.info("Start preprocessing...")
        DataMonitor.check_missing_values(df)

        df["date_time"] = pd.to_datetime(df["date_time"])
        df["hour"] = df["date_time"].dt.hour

        X = df[["hour"]]
        y = df["traffic_volume"]

        DataMonitor.log_data_shape("Preprocessing Features", X)
        return train_test_split(X, y, test_size=0.2, shuffle=False)

### 2.2 PySpark: Schaalbaarheid naar productie (Leerdoel 2)

De huidige `DataIngestor` gebruikt Pandas met `chunksize=10000`. Dit is afdoende voor de Metro Interstate Traffic Volume dataset (~48.000 rijen op één machine). In een productieomgeving waarbij sensoren van tientallen kruispunten continu data aanleveren, groeit de dataset naar honderden miljoenen rijen. Op dat punt wordt het geheugen van één machine een bottleneck.

PySpark lost dit op door data-ingestie en feature engineering te distribueren over een cluster van machines via Apache Spark. De cel hieronder toont hoe de ingestie er op een cloud-cluster (bijv. Azure Databricks of AWS EMR) uit zou zien. De code is uitgecommentarieerd: uitvoering vereist een actieve Spark-omgeving. De Pandas-implementatie in `app.py` blijft de actieve oplossing voor dit prototype.

In [4]:
# Proof-of-concept: PySpark data ingestion voor productie op een cloud-cluster.
# Uitvoering vereist een actieve Spark-omgeving (bijv. Azure Databricks of AWS EMR).
# De Pandas DataIngestor in app.py blijft actief in dit prototype.

# from pyspark.sql import SparkSession
# from pyspark.sql.functions import hour as spark_hour, col
#
#
# class SparkDataIngestor:
#     """
#     Vervangt DataIngestor bij datasets groter dan het geheugen van één machine.
#     Distribueert data-ingestie en feature engineering over een Spark-cluster.
#     """
#
#     def __init__(self, data_path: str):
#         self.data_path = data_path
#         self.spark = SparkSession.builder \
#             .appName("NovaStad-TrafficIngestion") \
#             .getOrCreate()
#
#     def load_data(self):
#         df_spark = self.spark.read.csv(
#             self.data_path,
#             header=True,
#             inferSchema=True
#         )
#         # Feature engineering direct in Spark, verdeeld over het cluster
#         df_spark = df_spark.withColumn("hour", spark_hour(col("date_time")))
#         df_spark = df_spark.select("hour", "traffic_volume").dropna()
#
#         row_count = df_spark.count()
#         print(f"Totaal rijen verwerkt door cluster: {row_count}")
#
#         # Sla gefilterde features op als Parquet voor efficiënte herlezing
#         df_spark.write.mode("overwrite").parquet("hdfs:///nova_stad/features/")
#
#         # Converteer naar Pandas voor scikit-learn compatibiliteit
#         return df_spark.toPandas()

print("SparkDataIngestor proof-of-concept gereed (niet uitgevoerd: vereist Spark-cluster).")
print("Op een productie-cluster vervangt SparkDataIngestor de Pandas DataIngestor.")
print("Huidige prototype-implementatie: Pandas met chunksize=10000.")

SparkDataIngestor proof-of-concept gereed (niet uitgevoerd: vereist Spark-cluster).
Op een productie-cluster vervangt SparkDataIngestor de Pandas DataIngestor.
Huidige prototype-implementatie: Pandas met chunksize=10000.


## 3. Modellering (Leerdoel 3)

De pipeline maakt gebruik van twee modellen:

**Cloud Model**
Een `RandomForestRegressor` die het verkeersvolume voorspelt op basis van het uur van de dag. Dit model draait op een server en verwerkt historische data.

**Edge Model**
Een pre-trained `SSDLite MobileNetV3` dat objecten herkent in afbeeldingen: autos, bussen, vrachtwagens, fietsen, motoren en voetgangers. Dit model is specifiek ontworpen voor edge-devices met beperkt rekenvermogen en geheugen.

In [5]:
import mlflow
import mlflow.sklearn
import torch
from torchvision.models.detection import (
    ssdlite320_mobilenet_v3_large,
    SSDLite320_MobileNet_V3_Large_Weights
)
from PIL import Image
import io


class CloudModel:
    """Cloud Model: Verkeersvoorspelling op basis van historische data."""

    def __init__(self):
        self.model = RandomForestRegressor(random_state=42, n_estimators=50)
        self.is_trained = False

    def train(self, X_train: pd.DataFrame, y_train: pd.Series):
        logger.info("Start training Cloud Model (RandomForestRegressor)...")
        self.model.fit(X_train, y_train)
        self.is_trained = True
        logger.info("Cloud Model training voltooid.")

    def evaluate(self, X_test: pd.DataFrame, y_test: pd.Series):
        if not self.is_trained:
            logger.error("Model is nog niet getraind.")
            return
        preds = self.model.predict(X_test)
        rmse = np.sqrt(mean_squared_error(y_test, preds))
        logger.info(f"Cloud Model Evaluatie - RMSE: {rmse:.2f}")

        # MLflow experiment tracking
        mlflow.set_experiment("nova_stad_traffic_prediction")
        with mlflow.start_run():
            mlflow.log_param("n_estimators", self.model.n_estimators)
            mlflow.log_param("random_state", self.model.random_state)
            mlflow.log_metric("rmse", round(rmse, 2))
            mlflow.sklearn.log_model(self.model, "random_forest_model")
        logger.info(
            "Experiment gelogd via MLflow (experiment: nova_stad_traffic_prediction)"
        )

    def predict(self, hour: int) -> float:
        if not self.is_trained:
            raise ValueError("Model is niet getraind.")
        return float(self.model.predict([[hour]])[0])


class EdgeModel:
    """Edge Model: Lichtgewicht objectdetectie via PyTorch MobileNet."""

    def __init__(self):
        logger.info("Laden van pre-trained Edge Model (SSDLite MobileNetV3)...")
        self.weights = SSDLite320_MobileNet_V3_Large_Weights.DEFAULT
        self.model = ssdlite320_mobilenet_v3_large(weights=self.weights)
        self.model.eval()
        self.preprocess = self.weights.transforms()
        self.categories = self.weights.meta["categories"]
        logger.info("Edge Model succesvol geladen.")

    def predict(self, image_bytes: bytes) -> dict:
        image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
        batch = [self.preprocess(image)]

        with torch.no_grad():
            prediction = self.model(batch)[0]

        results = []
        for i, score in enumerate(prediction["scores"]):
            if score > 0.5:
                label_idx = prediction["labels"][i].item()
                label_name = self.categories[label_idx]
                if label_name in ["car", "person", "bus", "truck", "bicycle", "motorcycle"]:
                    results.append({
                        "object": label_name,
                        "score": round(score.item(), 2)
                    })

        logger.info(
            f"Edge Model detectie voltooid. Aantal relevante objecten: {len(results)}"
        )
        return {"detections": results, "count": len(results)}


### 3.2 Modelkeuze — Verantwoording (Leerdoel 3)

**Cloud Model: RandomForestRegressor**

De keuze voor RandomForest is gebaseerd op de eigenschappen van de verkeersdata. Het verkeersvolume kent niet-lineaire patronen: ochtend- en avondspits, weekendverschillen en incidentele uitschieters door wegwerkzaamheden of evenementen. RandomForest middelt de uitkomst van meerdere decision trees, waardoor individuele uitschieters minder zwaar wegen. Het model presteert goed op tabular data met dit type niet-lineaire structuur zonder uitgebreide feature engineering. De hyperparameterkeuze wordt onderbouwd in sectie 4.1.

**Edge Model: SSDLite320 MobileNetV3**

SSDLite320 MobileNetV3 is ontworpen voor CPU-inferentie op hardware met beperkt rekenvermogen. De SSDLite-variant vervangt standaard convoluties door depthwise separable convoluties, wat de rekenbelasting sterk verlaagt ten opzichte van standaard SSD zonder significant precisieverlies voor objectdetectie. De pre-trained COCO-gewichten bevatten alle relevante objectklassen voor dit project (car, bus, truck, bicycle, motorcycle, person). Extra trainingsdata van Nova Stad-camera's is voor het huidige prototype niet vereist. Op standaard CPU-hardware ligt de inferentiesnelheid onder de 2 seconden per afbeelding, in lijn met de gestelde prestatievereiste.

## 4. Pipeline Uitvoering

Hieronder wordt de volledige pipeline stap voor stap uitgevoerd: data inladen via URL, preprocessing, training en evaluatie van het Cloud Model.

In [6]:
DATA_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00492/Metro_Interstate_Traffic_Volume.csv.gz"

ingestor = DataIngestor(DATA_URL)
preprocessor = Preprocessor()
cloud_model = CloudModel()
edge_model = EdgeModel()

raw_data = ingestor.load_data()
X_train, X_test, y_train, y_test = preprocessor.process(raw_data)
cloud_model.train(X_train, y_train)
cloud_model.evaluate(X_test, y_test)

2026-06-21 22:25:59,945 - INFO - Laden van pre-trained Edge Model (SSDLite MobileNetV3)...
2026-06-21 22:26:00,207 - INFO - Edge Model succesvol geladen.
2026-06-21 22:26:00,209 - INFO - Start inladen van data via URL...
2026-06-21 22:26:02,551 - INFO - [Data Ingestion] Data dimensies: (48204, 9)
2026-06-21 22:26:02,570 - INFO - Statistieken opgeslagen in monitoring_metrics.csv
2026-06-21 22:26:02,571 - INFO - Start preprocessing...
2026-06-21 22:26:02,576 - WARNING - Let op: 48143 ontbrekende waarden gedetecteerd in de dataset.
2026-06-21 22:26:02,612 - INFO - [Preprocessing Features] Data dimensies: (48204, 1)
2026-06-21 22:26:02,617 - INFO - Start training Cloud Model (RandomForestRegressor)...
2026-06-21 22:26:02,933 - INFO - Cloud Model training voltooid.
2026-06-21 22:26:02,952 - INFO - Cloud Model Evaluatie - RMSE: 953.18
2026/06/21 22:26:04 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/21 22:26:04 INFO mlflow.store.db.utils: Updating database ta

### 4.1 Hyperparameter Tuning via GridSearchCV (Leerdoel 3)

De `RandomForestRegressor` in de pipeline gebruikt `n_estimators=50` en `max_depth=None` (standaard). De cel hieronder voert een `GridSearchCV` uit over een beperkte parameterruimte en toont dat de gekozen waarden overeenkomen met de beste resultaten uit de zoekopdracht. Dit valideert de parameterkeuze in `app.py` op basis van data.

In [7]:
from sklearn.model_selection import GridSearchCV

# Parameterruimte voor de zoekopdracht
param_grid = {
    "n_estimators": [10, 50, 100],
    "max_depth": [None, 5, 10]
}

rf_base = RandomForestRegressor(random_state=42)

grid_search = GridSearchCV(
    estimator=rf_base,
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=3,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

best_params = grid_search.best_params_
best_rmse = -grid_search.best_score_

print(f"Beste parameters gevonden door GridSearchCV: {best_params}")
print(f"Beste cross-validated RMSE:                  {best_rmse:.2f}")
print()
print(f"Pipeline-waarde n_estimators=50 gevalideerd: {best_params['n_estimators'] == 50}")
print(f"Pipeline-waarde max_depth=None gevalideerd:  {best_params['max_depth'] is None}")

Fitting 3 folds for each of 9 candidates, totalling 27 fits
Beste parameters gevonden door GridSearchCV: {'max_depth': None, 'n_estimators': 100}
Beste cross-validated RMSE:                  944.78

Pipeline-waarde n_estimators=50 gevalideerd: False
Pipeline-waarde max_depth=None gevalideerd:  True


## 5. Deployment (Leerdoel 4)

Voor deployment gebruiken we FastAPI voor de ontsluiting van de modellen en GitHub Actions voor de Continuous Integration (CI).

### GitHub Actions (.github/workflows/ci.yml)
GitHub Actions voert automatisch een kwaliteitscontrole uit bij elke push naar de `main` branch. Dit proces installeert de benodigde libraries en voert een automatische py_compile test uit op `app.py`. Dit garandeert dat we geen syntaxfouten naar de productie-omgeving pushen. 

Docker-implementatie is meegenomen in de theorie-ontwerpen, maar is voor dit prototype buiten scope gelaten.

### FastAPI Applicatie

De volgende cel schrijft de FastAPI applicatie naar een bestand `app.py`. Als je `uvicorn.run()` direct in een notebookcel uitvoert, blokkeert het notebook omdat de server blijft draaien en de cel nooit afgerond wordt.

Start de API na het uitvoeren van de cel via de terminal met het volgende commando:

```
uvicorn app:app --reload
```

De API is daarna bereikbaar op `http://127.0.0.1:8000`. De automatische documentatie staat op `http://127.0.0.1:8000/docs`.

In [8]:
%%writefile app.py
import os
import io
import csv
from datetime import datetime
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
import logging
import torch
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from torchvision.models.detection import (
    ssdlite320_mobilenet_v3_large,
    SSDLite320_MobileNet_V3_Large_Weights
)
from PIL import Image
from fastapi import FastAPI, UploadFile, File
import joblib

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)


class DataMonitor:
    @staticmethod
    def log_data_shape(step_name, df):
        logger.info(f"[{step_name}] Data dimensies: {df.shape}")

    @staticmethod
    def check_missing_values(df):
        missing = df.isnull().sum().sum()
        if missing > 0:
            logger.warning(f"Let op: {missing} ontbrekende waarden gedetecteerd.")
        else:
            logger.info("Geen ontbrekende waarden gedetecteerd.")

    @staticmethod
    def trigger_alert(df):
        stats = {
            "timestamp": [pd.Timestamp.now().isoformat()],
            "rows": [df.shape[0]],
            "columns": [df.shape[1]],
            "missing_values": [int(df.isnull().sum().sum())],
            "mean_traffic_volume": [
                round(df["traffic_volume"].mean(), 2)
                if "traffic_volume" in df.columns else None
            ]
        }
        stats_df = pd.DataFrame(stats)
        file_exists = os.path.isfile("monitoring_metrics.csv")
        stats_df.to_csv(
            "monitoring_metrics.csv", mode="a", header=not file_exists, index=False
        )
        logger.info("Statistieken opgeslagen in monitoring_metrics.csv")


class DataIngestor:
    def __init__(self, url):
        self.url = url

    def load_data(self):
        chunks = pd.read_csv(self.url, chunksize=10000)
        df = pd.concat(chunks, ignore_index=True)
        DataMonitor.log_data_shape("Data Ingestion", df)
        DataMonitor.trigger_alert(df)
        return df


class Preprocessor:
    def process(self, df):
        DataMonitor.check_missing_values(df)
        df["date_time"] = pd.to_datetime(df["date_time"])
        df["hour"] = df["date_time"].dt.hour
        X = df[["hour"]]
        y = df["traffic_volume"]
        return train_test_split(X, y, test_size=0.2, shuffle=False)


class CloudModel:
    def __init__(self):
        self.model_path = "cloud_model.pkl"
        self.model = RandomForestRegressor(random_state=42, n_estimators=50)
        self.is_trained = False

        if os.path.exists(self.model_path):
            self.model = joblib.load(self.model_path)
            self.is_trained = True
            logger.info("Bestaand model geladen vanaf schijf.")

    def train(self, X_train, y_train):
        if not self.is_trained:
            logger.info("Nieuw model wordt getraind...")
            mlflow.set_experiment("nova_stad_traffic_prediction")
            with mlflow.start_run():
                mlflow.log_param("n_estimators", self.model.n_estimators)
                mlflow.log_param("random_state", self.model.random_state)
                self.model.fit(X_train, y_train)
                train_rmse = float(
                    np.sqrt(
                        mean_squared_error(y_train, self.model.predict(X_train))
                    )
                )
                mlflow.log_metric("train_rmse", round(train_rmse, 2))
                mlflow.sklearn.log_model(self.model, "random_forest_model")
            joblib.dump(self.model, self.model_path)
            self.is_trained = True
            logger.info(f"Model succesvol opgeslagen als {self.model_path}")

    def predict(self, hour):
        if not self.is_trained:
            raise ValueError("Model is niet getraind.")
        return float(self.model.predict([[hour]])[0])


class EdgeModel:
    def __init__(self):
        self.weights = SSDLite320_MobileNet_V3_Large_Weights.DEFAULT
        self.model = ssdlite320_mobilenet_v3_large(weights=self.weights)
        self.model.eval()
        self.preprocess = self.weights.transforms()
        self.categories = self.weights.meta["categories"]

    def predict(self, image_bytes):
        image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
        batch = [self.preprocess(image)]
        with torch.no_grad():
            prediction = self.model(batch)[0]
        results = []
        for i, score in enumerate(prediction["scores"]):
            if score > 0.5:
                label_idx = prediction["labels"][i].item()
                label_name = self.categories[label_idx]
                # Bounding box coördinaten [x_min, y_min, x_max, y_max]
                box = prediction["boxes"][i].tolist()
                if label_name in ['car', 'person', 'bus', 'truck', 'bicycle', 'motorcycle']:
                    results.append({
                        "object": label_name,
                        "score": round(score.item(), 2),
                        "box": [round(b, 2) for b in box]
                    })
        return {"detections": results, "count": len(results)}


DATA_URL = (
    "https://archive.ics.uci.edu/ml/machine-learning-databases/"
    "00492/Metro_Interstate_Traffic_Volume.csv.gz"
)

ingestor = DataIngestor(DATA_URL)
preprocessor = Preprocessor()
cloud_model = CloudModel()
edge_model = EdgeModel()

if not getattr(cloud_model, "is_trained", False):
    logger.info("Geen bestaand model gevonden. Koude start: data inladen en trainen...")
    raw_data = ingestor.load_data()
    X_train, X_test, y_train, y_test = preprocessor.process(raw_data)
    cloud_model.train(X_train, y_train)

app = FastAPI(
    title="Nova Stad MLOps API",
    description="API voor Edge en Cloud modellen."
)


@app.get("/health")
def health_check():
    return {
        "status": "online",
        "cloud_model_trained": getattr(cloud_model, "is_trained", False)
    }


@app.get("/cloud/predict_traffic")
def predict_traffic(hour: int):
    if hour < 0 or hour > 23:
        return {"error": "Uur moet tussen 0 en 23 liggen."}
    try:
        prediction = cloud_model.predict(hour)
        logger.info(f"Cloud API request voor uur {hour}: voorspelling {prediction:.2f}")
        file_exists = os.path.isfile("api_live_logs.csv")
        with open("api_live_logs.csv", mode="a", newline="") as file:
            writer = csv.writer(file)
            if not file_exists:
                writer.writerow(["timestamp", "hour", "predicted_traffic_volume"])
            writer.writerow([datetime.now().isoformat(), hour, round(prediction, 2)])
        return {"hour": hour, "predicted_traffic_volume": prediction}
    except Exception as e:
        logger.error(f"Fout in Cloud API: {e}")
        return {"error": str(e)}


@app.post("/edge/detect_objects")
def detect_objects(file: UploadFile = File(...)):
    try:
        contents = file.file.read()
        results = edge_model.predict(contents)
        return {"filename": file.filename, "results": results}
    except Exception as e:
        logger.error(f"Fout in Edge API: {e}")
        return {"error": str(e)}


Writing app.py
